# RF Arcade · demo-night setup

**ESP32 SDR → Agilex 3 FPGA → CyberEther**

**What is added:** a fullscreen presentation and automatic seven-scene showreel using checkpoint 5.

**Why it is useful:** move from real airwaves to clean tone rescue and visible clipping with a few buttons.

**How it works:** ESP32 bursts or an FPGA test generator feed capture RAM, frequency translation, FIR II, decimation and gain/meters. Nios V applies settings; the host verifies results and renders the display FFT and CyberEther graphics.

```text
ESP32 RF ───────┐
               ├→ FPGA RAM → NCO → FIR II → decimate → gain / meters → CyberEther
FPGA test IQ ──┘                       ↑ Nios V / JTAG control
```

This is a presentation of **checkpoint 5**, not another FPGA design. No ESP32 reflashing is needed with the compatible firmware already installed. Run cells in order through **Launch**, then stop there while presenting. Run the final **Stop** cell when finished; do not use Run All for a live presentation.

## 1 · Install once and select the kernel

In a terminal, from `esp32_agilex3_iq`, on the Linux host or container desktop **where Quartus is installed**:

```bash
# Optional: set QUARTUS_ROOTDIR, QUESTA_ROOT and ATUM_ALTERA_ROOT for your installation.
bash scripts/install_local.sh --system
.venv-notebooks/bin/jupyter lab notebooks --no-browser
```

Omit `--system` if system prerequisites are already installed. That option needs sudo. The installer creates the venv/kernel and downloads the pinned CyberEther runtime; install/license Quartus, device support, Questa and Nios separately. See [installation details](README.md#install-once).

Select **Atum A3 IQ Demos**, or `.venv-notebooks/bin/python` in VS Code. Keep the desktop's `DISPLAY`. Set UART and JTAG overrides below for your board; the current container uses `/dev/ttyUSB0`. No Docker-specific commands are required. Long-running cells show an activity bar, elapsed time and the current step/log. The bar fills on successful completion; it does not guess a tool percentage. These configuration checks do not open the serial device.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/lessons.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "cyberether"))
# Optional overrides, set before tool discovery:
# os.environ["QUARTUS_ROOTDIR"] = "/your/install/quartus"
# os.environ["QUESTA_ROOT"] = "/your/install/questa_fse"
# os.environ["ATUM_ALTERA_ROOT"] = "/your/install"
# os.environ["ATUM_UART"] = "/dev/serial/by-id/your-board"
# os.environ["ATUM_JTAG_CABLE"] = "your cable from jtagconfig"
from lessons import Lesson, lan_ip, stop_demo
from common import tool_exe
CHECKPOINT = 5
FORCE_REBUILD = False   # Fresh checkout builds; source-matched SOFs are reused afterwards.
lesson = Lesson(CHECKPOINT)
HOST = lan_ip()
print("Python:", sys.executable)
print("Quartus:", tool_exe("quartus_sh"))
print("Questa:", tool_exe("vsim"))
print("Browser host:", HOST)

# Override HOST with the address reachable from your browser when necessary.
# HOST = "192.168.x.x"
from check_tools import check
checks = check()


## 2 · See the FPGA design

![Checkpoint 5 · native Quartus VDS diagram](diagrams/05_vds_gui.png)

The saved PNG above comes from the actual VDS GUI. Optionally run the next cell to regenerate the IP/design and export a fresh native diagram locally. It also prints complete Quartus project and ZIP links, including SDC and generated sources. This step needs the GUI prerequisites and can take a few minutes; skip it for a quick launch from an existing build.

Timing constraints: [checkpoint-5 SDC](../fpga_hub/iq_bridge/vds_demo_05_gain/iq_bridge_vds.sdc).

In [ ]:
from IPython.display import Image, display
diagram_png = lesson.generate()
display(Image(filename=str(diagram_png)))
print("Exported:", diagram_png)

## 3 · Build or verify the checkpoint-5 SOF

A fresh checkout compiles the design. Later runs reuse only a source-matched, hash-verified SOF; set `FORCE_REBUILD=True` above for a full compile. This does not program the board. Use the printed QPF/project ZIP links to inspect or compile in Quartus; extract ZIPs first. The checkpoint includes the Nios boot monitor already. Generating a separate Hello World project is optional: [Nios IDE guide](../software/README.md).

For the Python model, module simulation and Questa GUI walkthrough, use [lesson 05](05_gain_power_meters.ipynb).

In [ ]:
build = lesson.build(force=FORCE_REBUILD)
print("SOF:", lesson.stage / "iq_bridge_vds.sof")
print("SHA256:", build["sha256"])
print("Compiled resources:", build["resources"])

## 4 · Program the physical Atum A3 Nano

Connect board power, JTAG and UART; keep the three compatible ESP32 modules connected. Stop unrelated acquisition owners first. This cell stops the previous notebook-owned demo and **programs checkpoint 5**, replacing the current FPGA image. No ESP32 reset-button presses are required with installed compatible firmware.

In [ ]:
lesson.program()

## 5 · Launch RF Arcade

The next cell starts actual acquisition and waits for a verified live result. Open its **private control-invite link**, then choose fullscreen. Use that link rather than opening port 8687 directly: it includes the controls and access token. Keep tokens private.

`HOST` must be reachable from the viewing browser. Default ports are **8768** (controls), **8687** (native viewer), and **TCP 40100–40103** (media). If using a container or firewall, expose those ports and set `HOST` to a reachable address. A launch failure is an error to investigate, not evidence of live samples.

**Stop executing cells here once the link opens.** Leave the notebook kernel running while presenting.

In [ ]:
live = lesson.start_demo(host=HOST, demo_night=True)
print(live)

## Try this two-minute demo

| Scene | Try | Point out |
|---|---|---|
| **Airwaves** | Select radio and Wi-Fi channel | Real received 2.4 GHz bursts, waterfall and IQ samples |
| **Rescue the tone** | Fold → Shift to DC → Reject interferer | FPGA mixing, anti-alias filtering and decimation |
| **Power rush** | 1× → 2× → 16× | Amplitude growth, clipping, harmonics and FPGA meters |
| **Showreel** | Start, then fullscreen | Seven scenes, each waiting for verified hardware settings |

**Freeze / Resume** pauses/restarts acquisition. Manual scene, radio or channel changes stop the showreel. Stop showreel leaves the selected scene active.

Live Airwaves uses RF; tone rescue and power rush use clean **FPGA-generated test signals**, with no RF transmission. The IQ orbit shows sample scatter, not recovered Wi-Fi/QPSK symbols. Power is relative, not calibrated dBm. Each waterfall row is a distinct sparse capture, not a gap-free recording. Radio selection uses one ESP32 at a time, without phase coherence. Host transport, integer reference checks and display FFT remain host computations.

More detail: [RF Arcade guide](../docs/RF_ARCADE.md).

## 6 · Stop and release the hardware

Run this cell **when finished presenting**. It stops this checkout's notebook-owned demo and its acquisition/JTAG children, releasing their device handles. It does not stop unrelated user processes or erase the FPGA image. To present again, rerun Launch; to switch designs, open another lesson and run its programming cell.

In [ ]:
stop_demo()
print("Notebook-owned RF Arcade stopped; its UART/JTAG handles are released.")

### CyberEther · example screenshot

Saved browser screenshot from a physical Atum A3 Nano / Agilex 3 hardware test. This is a static example; run the launch cell above for current data.

RF Arcade’s Power rush scene at 16× gain, using an FPGA-generated test signal. The display shows saturation, harmonics, clipping and zero integer-reference error.

![CyberEther example — lesson 06](screenshots/06_cyberether.png)

[Open full-resolution screenshot](screenshots/06_cyberether.png).